# 03 · Funding over time

**Goal:** Understand how Northeastern's grant funding has evolved over time (2000–2025).

Sections:
1. Setup & data load
2. Annual grant count & cumulative trend
3. Annual funding volume ($ total) & rolling averages
4. Award count vs total dollars (volume vs size over time)
5. Funding by college × year heatmap
6. Agency mix over time
7. Pre/post-COVID analysis (pre: 2015–2019, post: 2020–2024)
8. Observations & key findings

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid', palette='muted')

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'data' / 'processed').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
OUTPUTS_DIR   = REPO_ROOT / 'outputs'
OUTPUTS_DIR.mkdir(exist_ok=True)

In [ ]:
faculty       = pd.read_parquet(PROCESSED_DIR / 'faculty.parquet')
grants        = pd.read_parquet(PROCESSED_DIR / 'grants.parquet')
faculty_grants = pd.read_parquet(PROCESSED_DIR / 'faculty_grants.parquet')

# Year column guard
assert 'startdateyear' in grants.columns, "startdateyear column missing from grants"

# Filter to valid year range
grants_yr = grants[(grants['startdateyear'] >= 2000) &
                   (grants['startdateyear'] <= 2025)].copy()
print(f'grants in 2000–2025: {len(grants_yr):,} of {len(grants):,}')

## 1 · Annual Grant Count & Cumulative Trend

In [ ]:
annual = (grants_yr
          .groupby('startdateyear')
          .agg(n_grants=('grant_id', 'nunique'),
               total_dollars=('totaldollars', 'sum'),
               avg_dollars=('totaldollars', 'mean'),
               median_dollars=('totaldollars', 'median'))
          .reset_index()
          .rename(columns={'startdateyear': 'year'}))

annual['cumulative_grants']  = annual['n_grants'].cumsum()
annual['cumulative_dollars'] = annual['total_dollars'].cumsum()
annual['rolling_3yr_count']  = annual['n_grants'].rolling(3, center=True).mean()
annual['rolling_3yr_dollars']= annual['total_dollars'].rolling(3, center=True).mean()

display(annual)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

# Panel 1: Annual count + rolling average
axes[0].bar(annual['year'], annual['n_grants'],
            color='steelblue', alpha=0.7, label='Annual grants')
axes[0].plot(annual['year'], annual['rolling_3yr_count'],
             color='darkred', linewidth=2, label='3-yr rolling avg')
axes[0].set_ylabel('Number of grants')
axes[0].set_title('Annual Grant Count (2000–2025)')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].axvspan(2020, 2022, alpha=0.1, color='red', label='COVID window')

# Panel 2: Cumulative grant count
axes[1].fill_between(annual['year'], annual['cumulative_grants'],
                     color='steelblue', alpha=0.35)
axes[1].plot(annual['year'], annual['cumulative_grants'],
             color='steelblue', linewidth=2)
axes[1].set_ylabel('Cumulative grants')
axes[1].set_xlabel('Year')
axes[1].set_title('Cumulative Grant Count')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'w5_annual_grant_count.png', dpi=150, bbox_inches='tight')
plt.show()

### Grants awarded count by year (2000–2025):
- Growth phase (2000-2018): Rapid expansion from ~5 grants in 2000 to a peak of ~150 grants around 2018-2020
- Plateau (2015-2020): The rolling average plateaus around 140-150 grants/year, suggesting maturity
- Decline (2020-2025): Notable drop in recent years, with 2025 showing only ~80 grants - the lowest since ~2007

## 2 · Annual Funding Volume & Rolling Averages

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

axes[0].bar(annual['year'], annual['total_dollars'] / 1e6,
            color='darkorange', alpha=0.7, label='Annual total')
axes[0].plot(annual['year'], annual['rolling_3yr_dollars'] / 1e6,
             color='darkred', linewidth=2, label='3-yr rolling avg')
axes[0].set_ylabel('Total funding ($M)')
axes[0].set_title('Annual Grant Funding Volume ($ millions)')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].axvspan(2020, 2022, alpha=0.1, color='red')

axes[1].fill_between(annual['year'], annual['cumulative_dollars'] / 1e6,
                     color='darkorange', alpha=0.35)
axes[1].plot(annual['year'], annual['cumulative_dollars'] / 1e6,
             color='darkorange', linewidth=2)
axes[1].set_ylabel('Cumulative funding ($M)')
axes[1].set_xlabel('Year')
axes[1].set_title('Cumulative Grant Funding')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'w5_annual_funding_volume.png', dpi=150, bbox_inches='tight')
plt.show()

## 3 · Award Volume vs Average Award Size

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 5))

ax2 = ax1.twinx()
ax1.bar(annual['year'], annual['n_grants'], color='steelblue', alpha=0.5, label='# grants (left)')
ax2.plot(annual['year'], annual['avg_dollars'] / 1e6, color='darkred',
         linewidth=2, marker='o', markersize=4, label='Avg grant size (right)')

ax1.set_xlabel('Year')
ax1.set_ylabel('Number of grants', color='steelblue')
ax2.set_ylabel('Average grant size ($M)', color='darkred')
ax1.set_title('Volume vs. Average Size Over Time')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'w5_volume_vs_size.png', dpi=150, bbox_inches='tight')
plt.show()

## 4 · Funding by College × Year Heatmap

In [ ]:
# Join faculty_grants → grants → faculty to get college per grant per year
gf_yr = (faculty_grants
         .merge(grants_yr[['grant_id', 'startdateyear', 'totaldollars']],
                on='grant_id', how='inner')
         .merge(faculty[['faculty_id', 'superior_academic_unit']],
                on='faculty_id', how='left'))

college_yr = (gf_yr
              .groupby(['superior_academic_unit', 'startdateyear'], observed=True)
              ['totaldollars']
              .sum()
              .unstack(fill_value=0))

# Keep top 10 colleges by total funding
top10_colleges = college_yr.sum(axis=1).sort_values(ascending=False).head(10).index
heat_data = college_yr.loc[top10_colleges]

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(heat_data / 1e6, cmap='YlOrRd', ax=ax,
            linewidths=0.4, annot=False,
            cbar_kws={'label': 'Funding ($M)'})
ax.set_title('Annual Grant Funding by College (Top 10, $M)')
ax.set_xlabel('Year')
ax.set_ylabel('')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'w5_college_year_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5 · Agency Mix Over Time

In [ ]:
# Top 5 agencies by count
top5_agencies = (grants['agencycode'].value_counts().head(5).index.tolist()
                 if 'agencycode' in grants.columns
                 else grants['agencyname'].value_counts().head(5).index.tolist())
a_col = 'agencycode' if 'agencycode' in grants.columns else 'agencyname'

agency_yr = (grants_yr[grants_yr[a_col].isin(top5_agencies)]
             .groupby([a_col, 'startdateyear'])
             ['grant_id'].nunique()
             .unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(13, 5))
agency_yr.T.plot(kind='area', stacked=True, ax=ax,
                 colormap='tab10', alpha=0.75)
ax.set_xlabel('Year')
ax.set_ylabel('Number of grants')
ax.set_title('Grant Count by Agency Over Time (Top 5 agencies)')
ax.legend(title='Agency', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'w5_agency_mix_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

### Agency counts by year (top 5 agencies):
- NSF (olive) dominates consistently maintaining strong presence
- NIH (pink) substantial presence throughout, espectially 2005-2015. More stable compared to NSF's fluctuations
- Heavy federal research dependence: NSF + NIH account for ~70-80% of all grants. Diversification is limited
- Recent decline affects all agencies: The 2020-2025 drop impacts the full stack, not just one funder
- 2005 spike anomaly: Sharp peak around 2005 appears to be driven by a temporary surge across multiple agencies

## 6 · Pre/Post-COVID Analysis

In [ ]:
windows = {
    'Pre-COVID (2015–2019)': (2015, 2019),
    'COVID (2020–2021)':     (2020, 2021),
    'Post-COVID (2022–2025)':(2022, 2025),
}

rows = []
for label, (y0, y1) in windows.items():
    g = grants_yr[(grants_yr['startdateyear'] >= y0) & (grants_yr['startdateyear'] <= y1)]
    rows.append({
        'Period': label,
        'Years': f'{y0}–{y1}',
        'n_grants': g['grant_id'].nunique(),
        'total_dollars': g['totaldollars'].sum(),
        'avg_per_year': g['totaldollars'].sum() / (y1 - y0 + 1),
        'avg_grant_size': g['totaldollars'].mean(),
        'median_grant_size': g['totaldollars'].median(),
    })

covid_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colors = ['steelblue', 'salmon', 'mediumseagreen']

axes[0].bar(covid_df['Period'], covid_df['n_grants'], color=colors)
axes[0].set_title('Total Grant Count')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(covid_df['Period'], covid_df['avg_per_year'] / 1e6, color=colors)
axes[1].set_title('Avg Annual Funding ($M)')
axes[1].tick_params(axis='x', rotation=20)

axes[2].bar(covid_df['Period'], covid_df['avg_grant_size'] / 1e3, color=colors)
axes[2].set_title('Avg Grant Size ($K)')
axes[2].tick_params(axis='x', rotation=20)

plt.suptitle('Pre / During / Post-COVID Grant Activity', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'w5_covid_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

for col in ['total_dollars', 'avg_per_year', 'avg_grant_size', 'median_grant_size']:
    covid_df[col] = covid_df[col].apply(lambda x: f'${x:,.0f}')
display(covid_df)

## 7 · Observations

Document key findings here

In [ ]:
# Save annual summary for downstream use
annual.to_csv(OUTPUTS_DIR / 'annual_grant_summary.csv', index=False)
print('Saved annual_grant_summary.csv')
annual.head()

# NOTES
- Faculty who moved between universities bring in the grants they were awarded at their previous institution, which can create a misleading picture of funding trends if not accounted for -— so we will only consider grants they earned in the last three years before they joined Northeastern. This way, we can better understand the funding landscape as it relates to Northeastern's current faculty and their recent research activities.